Section 1: Mount Google Drive



In [9]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Section 2: Extract Dataset

In [13]:
import zipfile
import os

zip_path = '/content/drive/MyDrive/Data_Science/bus-stop-assess final.v2i.yolov8.zip'
extract_path = '/content/dataset'

if not os.path.exists(extract_path):
    os.makedirs(extract_path)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset copy and unzip from Drive completed.!")

Dataset copy and unzip from Drive completed.!


In [14]:
import zipfile
import os

# Define the path to your zip file in Google Drive
zip_path = '/content/drive/MyDrive/Data_Science/bus-stop-assess final.v2i.yolov8.zip'
# Define where to extract the dataset
extract_path = '/content/dataset'

# Create the directory if it doesn't exist
if not os.path.exists(extract_path):
    os.makedirs(extract_path)

# Unzip the dataset into the local Colab directory
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset extraction from Drive completed!")

Dataset extraction from Drive completed!


Section 3: Configure data.yaml

In [ ]:
import yaml

yaml_path = '/content/dataset/data.yaml'

with open(yaml_path, 'r') as f:
    data = yaml.safe_load(f)

data['train'] = '/content/dataset/train/images'
data['val'] = '/content/dataset/valid/images'
data['test'] = '/content/dataset/test/images'

with open(yaml_path, 'w') as f:
    yaml.dump(data, f)

Section 4: Training the Model (Weights & Training)

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11n.pt')

model.train(
    data='/content/dataset/data.yaml',
    epochs=100,
    imgsz=640,
    project='/content/drive/MyDrive/Thesis_Project/training_results',
    name='bus_stop_v1',

    scale=0.5,
    perspective=0.0005,
    mosaic=1.0,
    mixup=0.2,
    degrees=10.0
)

Ultralytics 8.4.31 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset/data.yaml, degrees=10.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.2, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=bus_stop_v1, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspe

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4, 5])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7a18c0371be0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
     

Section 5: Model Inference (Testing on New Images)

After training, we use the best-trained weights (best.pt) to run predictions on the test dataset to visualize how the model performs on unseen data.

In [ ]:
import os
from ultralytics import YOLO

# 1. Load the best weights saved during training
model_path = '/content/drive/MyDrive/Thesis_Project/training_results/bus_stop_v1/weights/best.pt'
model = YOLO(model_path)

# 2. Define the path for test images
test_images_path = '/content/dataset/test/images'

# 3. Run prediction on the test dataset
if os.path.exists(test_images_path):
    results = model.predict(
        source=test_images_path,
        save=True,          # Save the predicted images with bounding boxes
        conf=0.25,          # Confidence threshold
        project='/content/drive/MyDrive/Thesis_Project',
        name='test_results'
    )
    print("Inference completed! Check the 'test_results' folder in your Drive.")
else:
    print("Error: Test images directory not found.")

Section 6: Quantitative Evaluation (Final Metrics)

This section calculates the final performance metrics (mAP, Precision, Recall) using the test split of your dataset for the thesis report.

In [ ]:
from ultralytics import YOLO

model_path = '/content/drive/MyDrive/Thesis_Project/training_results/bus_stop_v1/weights/best.pt'
model = YOLO(model_path)

metrics = model.val(data='/content/dataset/data.yaml', split='test')

print(f"mAP50: {metrics.box.map50:.3f}")
print(f"mAP50-95: {metrics.box.map:.3f}")
print(f"Precision: {metrics.box.mp:.3f}")
print(f"Recall: {metrics.box.mr:.3f}")

Ultralytics 8.4.32 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n summary (fused): 101 layers, 2,583,322 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1573.1±429.9 MB/s, size: 92.9 KB)
val: Scanning /content/dataset/test/labels... 118 images, 8 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 118/118 1.1Kit/s 0.1s
val: New cache created: /content/dataset/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 1.5it/s 5.2s
                   all        118        380      0.946      0.791       0.88      0.609
            route info         30         30      0.898      0.633      0.742        0.4
              schedule         33         33      0.945      0.848       0.94       0.71
               seating         71         72      0.892      0.804      0.886       0.58
               shelter         61         61      0.991      0.934       0.98      0.8